# Phase 1: 数据探索 & 特征工程

本 notebook 覆盖 Phase 1 的四个任务：
- **Task A**: 停电数据探索（目标变量）
- **Task B**: 天气特征探索 & 特征筛选
- **Task C**: 通用特征构造
- **Task D**: Baseline 复现 & 评估框架

每个人负责自己的部分，但建议大家都跑一遍，了解全貌。

In [ ]:
# ============== 安装 & 导入 ==============
# Colab 用户取消下面的注释
# !pip install netCDF4 xarray

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

# ====== 中文字体配置 ======
import matplotlib
import platform
if platform.system() == 'Darwin':  # macOS
    matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti']
elif platform.system() == 'Linux':  # Linux / Colab
    matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC', 'SimHei']
else:  # Windows
    matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 数据路径 — 确保 train.nc 已经放到 data/ 文件夹
DATA_DIR = "data/"
ds = xr.open_dataset(os.path.join(DATA_DIR, "train.nc"))

# 基本信息提取
timestamps = pd.to_datetime(ds.timestamp.values)
locations = list(ds.location.values)
weather_features = list(ds.feature.values) if 'feature' in ds.dims else []

# 核心数据矩阵
outage = ds.out.transpose("timestamp", "location").values.astype(float)      # (T, L)
tracked = ds.tracked.transpose("timestamp", "location").values.astype(float)  # (T, L)
weather = ds.weather.transpose("timestamp", "location", "feature").values.astype(float)  # (T, L, F)

T, L = outage.shape
F = len(weather_features)

print(f"时间范围: {timestamps.min()} ~ {timestamps.max()}")
print(f"时间步数: {T} (小时)")
print(f"县的数量: {L}")
print(f"天气特征数: {F}")
print(f"停电数据 shape: {outage.shape}")
print(f"天气数据 shape: {weather.shape}")

---
# Task A: 停电数据探索

**目标**：搞清楚停电数据长什么样——分布、哪些县严重、有没有时间规律。

## A1: 停电数据的整体分布

先看看 `out` 这个变量长什么样。剧透：绝大多数时间没停电（=0），偶尔有极端大值。

In [ ]:
# A1: 停电数据的基本统计
flat_out = outage.flatten()
print("=== 停电数据基本统计 ===")
print(f"总观测数: {len(flat_out):,}")
print(f"零值占比: {(flat_out == 0).mean():.1%}")
print(f"均值: {flat_out.mean():.2f}")
print(f"中位数: {np.median(flat_out):.2f}")
print(f"标准差: {flat_out.std():.2f}")
print(f"最大值: {flat_out.max():.0f}")
print(f"99%分位: {np.percentile(flat_out, 99):.0f}")
print(f"99.9%分位: {np.percentile(flat_out, 99.9):.0f}")

# 直方图：排除零值看分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：全部数据
axes[0].hist(flat_out, bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title("停电数分布 (全部数据)")
axes[0].set_xlabel("停电户数")
axes[0].set_ylabel("频次")
axes[0].set_yscale('log')

# 右图：只看非零值
nonzero = flat_out[flat_out > 0]
axes[1].hist(nonzero, bins=100, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title(f"停电数分布 (仅非零值, n={len(nonzero):,})")
axes[1].set_xlabel("停电户数")
axes[1].set_ylabel("频次")
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"\n结论: 数据极度右偏，{(flat_out==0).mean():.0%} 的观测值为0，极端值可达 {flat_out.max():.0f}。")
print("RMSE 对大值非常敏感，所以准确预测极端停电事件是拿分关键。")

## A2: 停电数据分层 — 数据驱动的阈值划分

A1 告诉我们数据极度稀疏且右偏。直接取平均会被极端值主导，掩盖日常模式。
所以先把数据分层，后续分析都基于这个分层框架。

**方法**：对 log 变换后的非零停电值做 **高斯混合模型 (GMM)**，让数据自动聚类成 3 簇，
避免人为选择分位数阈值的主观性。GMM 会找到数据分布中的自然断点。

In [ ]:
# A2: 停电数据分层 — GMM 数据驱动
from sklearn.mixture import GaussianMixture

flat_out = outage.flatten()
nonzero = flat_out[flat_out > 0]
log_nonzero = np.log1p(nonzero)  # log 变换让分布更接近高斯

# ====== GMM 自动聚类 ======
# 用 BIC 选最优簇数 (2~5)
bic_scores = {}
for k in range(2, 6):
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(log_nonzero.reshape(-1, 1))
    bic_scores[k] = gmm.bic(log_nonzero.reshape(-1, 1))

best_k = min(bic_scores, key=bic_scores.get)
print(f"=== GMM 最优簇数 (BIC 最低): k={best_k} ===")
for k, bic in bic_scores.items():
    marker = " ← 最优" if k == best_k else ""
    print(f"  k={k}: BIC={bic:,.0f}{marker}")

# 用最优 k 拟合
gmm = GaussianMixture(n_components=best_k, random_state=42)
gmm.fit(log_nonzero.reshape(-1, 1))
cluster_labels = gmm.predict(log_nonzero.reshape(-1, 1))

# 按均值排序簇 (0=最小=日常, ..., k-1=最大=极端)
cluster_means = [nonzero[cluster_labels == c].mean() for c in range(best_k)]
sorted_clusters = np.argsort(cluster_means)

# 名字和颜色池 — 支持 2~5 簇
_name_pool = {
    2: ['日常', '极端'],
    3: ['日常', '严重', '极端'],
    4: ['日常', '中等', '严重', '极端'],
    5: ['轻微', '日常', '中等', '严重', '极端'],
}
_color_pool = {
    2: ['#4CAF50', '#F44336'],
    3: ['#4CAF50', '#FF9800', '#F44336'],
    4: ['#4CAF50', '#2196F3', '#FF9800', '#F44336'],
    5: ['#8BC34A', '#4CAF50', '#2196F3', '#FF9800', '#F44336'],
}
regime_names = _name_pool.get(best_k, [f'层级{i+1}' for i in range(best_k)])
colors_gmm = _color_pool.get(best_k, plt.cm.viridis(np.linspace(0.2, 0.8, best_k)).tolist())

# 找到各簇之间的阈值（相邻簇 log 均值的中点，再转回原始尺度）
log_means_sorted = sorted(gmm.means_.flatten())
thresholds = []
for i in range(len(log_means_sorted) - 1):
    mid = (log_means_sorted[i] + log_means_sorted[i+1]) / 2
    thresholds.append(np.expm1(mid))

print(f"\n=== GMM 自动发现的阈值 ===")
for i, t in enumerate(thresholds):
    print(f"  {regime_names[i]} / {regime_names[i+1]} 分界: {t:.0f} 户")

# ====== 可视化: GMM 拟合结果 ======
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) log 分布 + GMM 各成分
ax = axes[0]
ax.hist(log_nonzero, bins=100, density=True, alpha=0.5, color='gray', label='实际分布')
x_range = np.linspace(log_nonzero.min(), log_nonzero.max(), 300)
for c in range(best_k):
    idx = sorted_clusters[c]
    mean_c = gmm.means_[idx, 0]
    std_c = np.sqrt(gmm.covariances_[idx, 0, 0])
    weight_c = gmm.weights_[idx]
    pdf = weight_c * (1/(std_c * np.sqrt(2*np.pi))) * np.exp(-0.5*((x_range - mean_c)/std_c)**2)
    ax.plot(x_range, pdf, linewidth=2, color=colors_gmm[c],
            label=f'{regime_names[c]} (μ={np.expm1(mean_c):.0f})')
# 标注阈值线
for t in thresholds:
    ax.axvline(np.log1p(t), color='black', linestyle='--', alpha=0.7)
ax.set_xlabel('log(1 + 停电户数)')
ax.set_ylabel('密度')
ax.set_title(f'GMM ({best_k}簇) 拟合非零停电分布')
ax.legend(fontsize=9)

# ====== 给所有数据打标签（含零值）======
def classify_outage_gmm(val):
    if val == 0:
        return '零值'
    for i, t in enumerate(thresholds):
        if val <= t:
            return regime_names[i]
    return regime_names[-1]

labels = np.array([classify_outage_gmm(v) for v in flat_out])
all_regimes = ['零值'] + regime_names
colors_map = {'零值': '#bdbdbd'}
for name, color in zip(regime_names, colors_gmm):
    colors_map[name] = color

# 2) 各层观测数量 vs 停电量贡献
counts = [np.sum(labels == r) for r in all_regimes]
sums = [flat_out[labels == r].sum() for r in all_regimes]

x_pos = np.arange(len(all_regimes))
width = 0.35
ax = axes[1]
ax_twin = ax.twinx()
bars1 = ax.bar(x_pos - width/2, [c/len(flat_out)*100 for c in counts], width,
               color=[colors_map.get(r, 'gray') for r in all_regimes], alpha=0.7, label='观测占比')
total_sum = flat_out.sum()
bars2 = ax_twin.bar(x_pos + width/2, [s/total_sum*100 for s in sums], width,
                    color=[colors_map.get(r, 'gray') for r in all_regimes], alpha=0.4,
                    edgecolor='black', linewidth=1.5, label='停电量占比')
ax.set_xticks(x_pos)
ax.set_xticklabels(all_regimes, fontsize=9)
ax.set_ylabel('观测占比 (%)')
ax_twin.set_ylabel('停电量占比 (%)')
ax.set_title('观测占比 vs 停电量贡献')
ax.legend(loc='upper left')
ax_twin.legend(loc='upper right')

# 3) 非零各层的分布
data_by_regime = [flat_out[labels == r] for r in regime_names]
bp = axes[2].boxplot(data_by_regime, labels=regime_names, patch_artist=True, showfliers=False)
for patch, r in zip(bp['boxes'], regime_names):
    patch.set_facecolor(colors_map.get(r, 'gray'))
axes[2].set_ylabel('停电户数')
axes[2].set_title('非零各层的分布 (不含离群值)')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

# ====== 统计表 ======
print(f"\n=== 各层统计 ===")
print(f"{'层级':>6s} | {'观测数':>10s} | {'占比':>7s} | {'均值':>10s} | {'中位数':>8s} | {'最大值':>10s} | {'占总停电量':>10s}")
print("-" * 80)
for regime in all_regimes:
    mask = labels == regime
    vals = flat_out[mask]
    n = len(vals)
    pct = n / len(flat_out) * 100
    mean_v = vals.mean() if n > 0 else 0
    med_v = np.median(vals) if n > 0 else 0
    max_v = vals.max() if n > 0 else 0
    sum_pct = vals.sum() / total_sum * 100 if total_sum > 0 else 0
    print(f"{'':>2s}{regime:>4s} | {n:>10,d} | {pct:>6.1f}% | {mean_v:>10.1f} | {med_v:>8.1f} | {max_v:>10.0f} | {sum_pct:>9.1f}%")

print(f"\n关键发现: GMM 数据驱动划分 (k={best_k})，阈值从分布形态中自动学到。")
print(f"  极端层虽然观测数少，但贡献了总停电量的大部分 → RMSE 优化重点。")

# 保存阈值供后续 cell 使用
gmm_thresholds = thresholds

## A3: 各县停电分析 — 热力图 + 地理映射

用两种方式看县级停电分布：
1. **时间-县 热力图**：直观看哪些县在什么时候停电最严重
2. **密歇根州地理热力图**：基于 FIPS 编码的 choropleth 地图

In [ ]:
# A3: 各县停电分析

# ====== 3.1 各县多维度统计 ======
# 用 GMM 阈值替代硬编码分位数
extreme_threshold = gmm_thresholds[-1]  # GMM 最高层阈值
severe_threshold = gmm_thresholds[0] if len(gmm_thresholds) > 1 else gmm_thresholds[0]

county_stats = pd.DataFrame({
    'fips': locations,
    'total_outage': outage.sum(axis=0),
    'mean_outage': outage.mean(axis=0),
    'max_outage': outage.max(axis=0),
    'mean_tracked': tracked.mean(axis=0),
    'nonzero_hours': (outage > 0).sum(axis=0),
    'severe_hours': (outage > severe_threshold).sum(axis=0),
    'extreme_hours': (outage > extreme_threshold).sum(axis=0),
})
county_stats['outage_rate'] = county_stats['mean_outage'] / county_stats['mean_tracked'].replace(0, np.nan)
county_stats['nonzero_pct'] = county_stats['nonzero_hours'] / T * 100
county_stats = county_stats.sort_values('total_outage', ascending=False).reset_index(drop=True)

print("=== 各县停电统计 Top 15 ===")
print(county_stats.head(15)[['fips', 'total_outage', 'max_outage', 'nonzero_pct',
                              'severe_hours', 'extreme_hours', 'mean_tracked']].to_string(index=False))

# ====== 3.2 时间-县 热力图 ======
daily_outage = pd.DataFrame(outage, index=timestamps).resample('D').sum().values
sorted_county_idx = county_stats['fips'].apply(lambda x: locations.index(x)).values

fig, ax = plt.subplots(figsize=(18, 10))
im = ax.imshow(np.log1p(daily_outage[:, sorted_county_idx].T),
               aspect='auto', cmap='YlOrRd', interpolation='nearest')
cbar = plt.colorbar(im, ax=ax, label='log(1 + 日停电总数)')

n_days = daily_outage.shape[0]
tick_step = max(n_days // 10, 1)
day_labels = pd.date_range(timestamps[0], periods=n_days, freq='D')
ax.set_xticks(range(0, n_days, tick_step))
ax.set_xticklabels([d.strftime('%m/%d') for d in day_labels[::tick_step]], rotation=45)
ax.set_yticks(range(0, len(locations), 5))
ax.set_yticklabels(county_stats['fips'].values[::5], fontsize=8)
ax.set_xlabel('日期')
ax.set_ylabel('县 (FIPS, 按总停电量排序)')
ax.set_title('各县每日停电热力图 — 颜色越深停电越严重')
plt.tight_layout()
plt.show()

print("热力图观察: 停电事件在时间和空间上高度集中 — 少数风暴日+少数县贡献了大部分停电。")

# ====== 3.3 密歇根州地理热力图 (plotly) ======
try:
    import plotly.express as px
    import plotly.io as pio
    from urllib.request import urlopen
    import json

    # 在 notebook 中正确渲染 plotly
    pio.renderers.default = 'notebook'

    with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
        counties_geojson = json.load(response)

    geo_df = county_stats[['fips', 'total_outage', 'max_outage', 'extreme_hours', 'outage_rate']].copy()
    geo_df['fips'] = geo_df['fips'].astype(str)
    geo_df['log_total'] = np.log1p(geo_df['total_outage'])

    # 图1: 总停电量
    fig1 = px.choropleth(
        geo_df, geojson=counties_geojson, locations='fips',
        color='log_total', color_continuous_scale='YlOrRd',
        scope='usa', title='密歇根州各县停电总量 (log scale)',
        labels={'log_total': 'log(1+总停电)'}
    )
    fig1.update_geos(fitbounds="locations")
    fig1.update_layout(width=700, height=500, margin=dict(l=0, r=0, t=40, b=0))
    fig1.show()

    # 图2: 极端停电小时数
    fig2 = px.choropleth(
        geo_df, geojson=counties_geojson, locations='fips',
        color='extreme_hours', color_continuous_scale='Reds',
        scope='usa', title='密歇根州各县极端停电小时数',
        labels={'extreme_hours': '极端停电(h)'}
    )
    fig2.update_geos(fitbounds="locations")
    fig2.update_layout(width=700, height=500, margin=dict(l=0, r=0, t=40, b=0))
    fig2.show()

    print("地理热力图: 观察停电严重的县在地理上是否有聚集性。")

except Exception as e:
    print(f"⚠ plotly 渲染失败: {e}")
    print("尝试保存为 HTML 文件，可以在浏览器中打开查看...")
    try:
        fig1.write_html("output/choropleth_total.html")
        fig2.write_html("output/choropleth_extreme.html")
        print("  已保存: output/choropleth_total.html, output/choropleth_extreme.html")
    except:
        print("  保存也失败了，请检查 plotly 安装: pip install plotly")

## A4: 分层时间模式分析 — 风暴时段 vs 日常时段

**核心改进**：
1. 风暴/日常划分：全州小时总停电 > P95 的连续时段（+ 6h 缓冲区）
2. 日常时段分析时**去掉零值**，只看"有停电发生的县-小时"，避免零值稀释模式
3. 风暴时段样本量小（~200h），加 **bootstrap 置信区间** 量化不确定性

In [ ]:
# A4: 分层时间模式分析

total_outage_by_time = outage.sum(axis=1)  # (T,) 每小时全州总停电

# ====== 4.1 识别风暴时段 ======
storm_threshold = np.percentile(total_outage_by_time, 95)
is_storm_raw = total_outage_by_time > storm_threshold

buffer_hours = 6
is_storm = is_storm_raw.copy()
for i in range(T):
    if is_storm_raw[i]:
        start = max(0, i - buffer_hours)
        end = min(T, i + buffer_hours + 1)
        is_storm[start:end] = True

is_quiet = ~is_storm
n_storm = is_storm.sum()
n_quiet = is_quiet.sum()

print(f"=== 时段划分 ===")
print(f"  风暴时段: {n_storm} 小时 ({n_storm/T*100:.1f}%)")
print(f"  日常时段: {n_quiet} 小时 ({n_quiet/T*100:.1f}%)")
print(f"  风暴阈值 (全州小时总停电 P95): {storm_threshold:.0f}")
print(f"  风暴时段停电占比: {total_outage_by_time[is_storm].sum()/total_outage_by_time.sum()*100:.1f}%")

# ====== 4.2 全时序图 + 风暴时段标注 ======
fig, ax = plt.subplots(figsize=(18, 5))
ax.plot(timestamps, total_outage_by_time, linewidth=0.6, color='black', alpha=0.7)

storm_starts, storm_ends = [], []
in_storm = False
for i in range(T):
    if is_storm[i] and not in_storm:
        storm_starts.append(i)
        in_storm = True
    elif not is_storm[i] and in_storm:
        storm_ends.append(i)
        in_storm = False
if in_storm:
    storm_ends.append(T - 1)

for s, e in zip(storm_starts, storm_ends):
    ax.axvspan(timestamps[s], timestamps[e], alpha=0.2, color='red')

peak_idx = np.argsort(total_outage_by_time)[-5:]
for idx in peak_idx:
    ax.annotate(f"{timestamps[idx].strftime('%m/%d %Hh')}\n{total_outage_by_time[idx]:,.0f}",
                xy=(timestamps[idx], total_outage_by_time[idx]),
                fontsize=8, color='red', ha='center')
ax.set_title(f'全州每小时总停电 (红色区域 = 风暴时段, 共 {len(storm_starts)} 次)')
ax.set_ylabel('停电户数')
plt.tight_layout()
plt.show()

# ====== 4.3 日常时段: 去掉零值后的日内模式 ======
# 关键: 把 outage (T, L) 在日常时段展开为 (county-hour) 级别，去掉零值
hours = timestamps.hour
hour_index = np.repeat(hours.values, L)  # 每个时间步重复 L 次 (对应 L 个县)
time_index = np.repeat(np.arange(T), L)
quiet_mask_expanded = np.repeat(is_quiet, L)

flat_outage = outage.flatten()  # (T*L,) — 按 (t0_l0, t0_l1, ..., t0_lL, t1_l0, ...)

# 日常 + 非零
quiet_nonzero_mask = quiet_mask_expanded & (flat_outage > 0)
quiet_nonzero_vals = flat_outage[quiet_nonzero_mask]
quiet_nonzero_hours = hour_index[quiet_nonzero_mask]

# 日常 + 含零 (对比用)
quiet_all_mask = quiet_mask_expanded
quiet_all_vals = flat_outage[quiet_all_mask]
quiet_all_hours = hour_index[quiet_all_mask]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 图1: 含零 vs 去零 对比 (日内)
ax = axes[0]
hourly_with_zero = pd.Series(quiet_all_vals).groupby(quiet_all_hours).mean()
hourly_no_zero = pd.Series(quiet_nonzero_vals).groupby(quiet_nonzero_hours).mean()
ax.bar(np.arange(24) - 0.2, hourly_with_zero.reindex(range(24), fill_value=0), 0.35,
       label='含零值 (被稀释)', color='#bdbdbd', alpha=0.8)
ax.bar(np.arange(24) + 0.2, hourly_no_zero.reindex(range(24), fill_value=0), 0.35,
       label='仅非零 (真实模式)', color='#4CAF50', alpha=0.8)
ax.set_xlabel('小时 (0-23)')
ax.set_ylabel('平均停电户数')
ax.set_title('日常时段日内模式: 含零 vs 去零')
ax.set_xticks(range(24))
ax.legend()

# 图2: 去零后箱线图
ax = axes[1]
quiet_by_hour_nz = []
for h in range(24):
    mask = (quiet_nonzero_hours == h)
    quiet_by_hour_nz.append(quiet_nonzero_vals[mask])
bp = ax.boxplot(quiet_by_hour_nz, positions=range(24), patch_artist=True, showfliers=False,
                boxprops=dict(facecolor='#4CAF50', alpha=0.6))
ax.set_xlabel('小时 (0-23)')
ax.set_ylabel('停电户数 (仅非零)')
ax.set_title('日常时段 (去零后) 日内分布')
ax.set_xticks(range(24))

# 图3: 去零后每小时的非零事件数量 (说明样本量)
ax = axes[2]
counts_per_hour = [len(quiet_by_hour_nz[h]) for h in range(24)]
ax.bar(range(24), counts_per_hour, color='#4CAF50', alpha=0.7)
ax.set_xlabel('小时 (0-23)')
ax.set_ylabel('非零观测数')
ax.set_title('日常时段每小时的非零样本量')
ax.set_xticks(range(24))
# 标注均值线
ax.axhline(np.mean(counts_per_hour), color='red', linestyle='--', label=f'均值: {np.mean(counts_per_hour):.0f}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"日常时段非零观测总数: {len(quiet_nonzero_vals):,}")
print(f"每小时平均样本量: ~{len(quiet_nonzero_vals)//24:,} → 样本量充足")

# ====== 4.4 风暴时段: 带 bootstrap 置信区间的星期/小时分析 ======
def bootstrap_ci(data, n_boot=1000, ci=95):
    """Bootstrap 均值的置信区间"""
    if len(data) < 3:
        return np.mean(data), np.mean(data), np.mean(data)
    boot_means = np.array([np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_boot)])
    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return np.mean(data), lower, upper

np.random.seed(42)
dow_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
dow = timestamps.dayofweek

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ---- 风暴: 按星期 (带 CI) ----
ax = axes[0, 0]
storm_dow_stats = []
for d in range(7):
    mask = is_storm & (dow == d)
    vals = total_outage_by_time[mask]
    mean, lo, hi = bootstrap_ci(vals)
    storm_dow_stats.append({'dow': d, 'mean': mean, 'lo': lo, 'hi': hi, 'n': len(vals)})
storm_dow_df = pd.DataFrame(storm_dow_stats)

ax.bar(range(7), storm_dow_df['mean'], color='#F44336', alpha=0.7)
ax.errorbar(range(7), storm_dow_df['mean'],
            yerr=[storm_dow_df['mean'] - storm_dow_df['lo'], storm_dow_df['hi'] - storm_dow_df['mean']],
            fmt='none', color='black', capsize=5, linewidth=2)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.set_ylabel('平均停电户数')
ax.set_title('风暴时段 — 星期模式 (带 95% bootstrap CI)')
# 标注样本量
for i, row in storm_dow_df.iterrows():
    ax.text(i, row['hi'] + row['mean']*0.02, f'n={row["n"]:.0f}', ha='center', fontsize=9, color='gray')

# ---- 风暴: 按小时 (带 CI) ----
ax = axes[0, 1]
storm_hour_stats = []
for h in range(24):
    mask = is_storm & (hours == h)
    vals = total_outage_by_time[mask]
    mean, lo, hi = bootstrap_ci(vals)
    storm_hour_stats.append({'hour': h, 'mean': mean, 'lo': lo, 'hi': hi, 'n': len(vals)})
storm_hour_df = pd.DataFrame(storm_hour_stats)

ax.bar(range(24), storm_hour_df['mean'], color='#F44336', alpha=0.7)
ax.errorbar(range(24), storm_hour_df['mean'],
            yerr=[storm_hour_df['mean'] - storm_hour_df['lo'], storm_hour_df['hi'] - storm_hour_df['mean']],
            fmt='none', color='black', capsize=3, linewidth=1.5)
ax.set_xticks(range(24))
ax.set_ylabel('平均停电户数')
ax.set_title('风暴时段 — 日内模式 (带 95% bootstrap CI)')
for i, row in storm_hour_df.iterrows():
    if i % 4 == 0:
        ax.text(i, row['hi'] + row['mean']*0.02, f'n={row["n"]:.0f}', ha='center', fontsize=8, color='gray')

# ---- 日常 (去零): 按星期 ----
ax = axes[1, 0]
quiet_nonzero_dow = dow.values[time_index[quiet_nonzero_mask] // 1]  # 用 time_index 还原 dow
# 更简单的方式: 直接用展开的 dow
dow_expanded = np.repeat(dow.values, L)
quiet_nz_dow = dow_expanded[quiet_nonzero_mask]

quiet_dow_stats = []
for d in range(7):
    vals = quiet_nonzero_vals[quiet_nz_dow == d]
    mean, lo, hi = bootstrap_ci(vals)
    quiet_dow_stats.append({'dow': d, 'mean': mean, 'lo': lo, 'hi': hi, 'n': len(vals)})
quiet_dow_df = pd.DataFrame(quiet_dow_stats)

ax.bar(range(7), quiet_dow_df['mean'], color='#4CAF50', alpha=0.7)
ax.errorbar(range(7), quiet_dow_df['mean'],
            yerr=[quiet_dow_df['mean'] - quiet_dow_df['lo'], quiet_dow_df['hi'] - quiet_dow_df['mean']],
            fmt='none', color='black', capsize=5, linewidth=2)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.set_ylabel('平均停电户数 (仅非零)')
ax.set_title('日常时段 (去零) — 星期模式 (带 95% bootstrap CI)')
for i, row in quiet_dow_df.iterrows():
    ax.text(i, row['hi'] + row['mean']*0.01, f'n={row["n"]:,.0f}', ha='center', fontsize=9, color='gray')

# ---- 日常 (去零): 按小时 ----
ax = axes[1, 1]
quiet_hour_stats = []
for h in range(24):
    vals = quiet_nonzero_vals[quiet_nonzero_hours == h]
    mean, lo, hi = bootstrap_ci(vals)
    quiet_hour_stats.append({'hour': h, 'mean': mean, 'lo': lo, 'hi': hi, 'n': len(vals)})
quiet_hour_df = pd.DataFrame(quiet_hour_stats)

ax.bar(range(24), quiet_hour_df['mean'], color='#4CAF50', alpha=0.7)
ax.errorbar(range(24), quiet_hour_df['mean'],
            yerr=[quiet_hour_df['mean'] - quiet_hour_df['lo'], quiet_hour_df['hi'] - quiet_hour_df['mean']],
            fmt='none', color='black', capsize=3, linewidth=1.5)
ax.set_xticks(range(24))
ax.set_ylabel('平均停电户数 (仅非零)')
ax.set_title('日常时段 (去零) — 日内模式 (带 95% bootstrap CI)')

plt.tight_layout()
plt.show()

# ====== 4.5 风暴事件清单 ======
print(f"\n=== 识别到的 {len(storm_starts)} 次风暴事件 ===")
print(f"{'编号':>4s} | {'起始时间':>16s} | {'结束时间':>16s} | {'持续(h)':>7s} | {'峰值停电':>10s} | {'累计停电':>12s}")
print("-" * 80)
for i, (s, e) in enumerate(zip(storm_starts, storm_ends)):
    duration = e - s
    peak = total_outage_by_time[s:e].max()
    total = total_outage_by_time[s:e].sum()
    print(f"{i+1:>4d} | {timestamps[s].strftime('%Y-%m-%d %H:%M'):>16s} | "
          f"{timestamps[min(e, T-1)].strftime('%Y-%m-%d %H:%M'):>16s} | "
          f"{duration:>7d} | {peak:>10,.0f} | {total:>12,.0f}")

# ====== 4.6 统计显著性总结 ======
print(f"\n=== 样本量 & 统计可靠性 ===")
print(f"  风暴总小时: {n_storm} → 每天约 {n_storm//7} 样本")
print(f"  日常非零观测: {len(quiet_nonzero_vals):,} → 每天约 {len(quiet_nonzero_vals)//7:,} 样本")
print(f"\n结论:")
print(f"  1. 风暴时段 (~{n_storm}h): CI 很宽 → 星期差异不显著，可能只是风暴碰巧落在某几天")
print(f"     不能说 '周X 停电更多'，样本不够支撑这个结论")
print(f"  2. 日常时段 (去零后, ~{len(quiet_nonzero_vals):,} 观测): 样本充足")
print(f"     如果 CI 窄且有趋势 → 可信的日内/星期模式 (可作为特征)")
print(f"  3. 停电的主驱动力是天气事件，不是时间周期 — 时间特征是辅助的")

---
# Task B: 天气特征探索 & 特征筛选

**目标**：从 109 个天气变量里找出和停电最相关的，删掉冗余的，缩减到可用的特征子集。

**改进思路**（基于 Part A 发现）：
- 数据极度稀疏 + 右偏 → Pearson 相关会被零值/极端值扭曲 → 加 Spearman 排名相关
- 风暴 vs 日常是两种不同数据生成过程 → 分层算相关性
- 线性相关可能漏掉非线性关系 → 加随机森林特征重要性

## B1: 多方法特征相关性排名

三种方法互相验证：
- **Pearson**：线性相关，对极端值敏感
- **Spearman**：排名相关，更鲁棒
- **分层**：风暴/日常时段分别算，看哪些特征在风暴时更重要

In [ ]:
# B1: 多方法特征相关性
from scipy.stats import spearmanr

flat_out = outage.flatten()
weather_flat = weather.reshape(-1, F)

# 构造分层 mask (展开到 T*L 维度)
storm_mask_expanded = np.repeat(is_storm, L)
quiet_mask_expanded = np.repeat(is_quiet, L)

# 对每个天气变量算四种相关性
results = []
for fi, feat_name in enumerate(weather_features):
    flat_w = weather_flat[:, fi]
    valid = ~(np.isnan(flat_out) | np.isnan(flat_w))

    if valid.sum() < 100:
        continue

    # 全局 Pearson
    pearson_r = np.corrcoef(flat_out[valid], flat_w[valid])[0, 1]

    # 全局 Spearman (采样加速，全量太慢)
    n_sample = min(200000, valid.sum())
    idx = np.random.choice(np.where(valid)[0], n_sample, replace=False)
    spearman_r, _ = spearmanr(flat_out[idx], flat_w[idx])

    # 风暴时段 Pearson
    storm_valid = valid & storm_mask_expanded
    if storm_valid.sum() > 100:
        storm_r = np.corrcoef(flat_out[storm_valid], flat_w[storm_valid])[0, 1]
    else:
        storm_r = np.nan

    # 日常非零 Pearson
    quiet_nz_valid = valid & quiet_mask_expanded & (flat_out > 0)
    if quiet_nz_valid.sum() > 100:
        quiet_r = np.corrcoef(flat_out[quiet_nz_valid], flat_w[quiet_nz_valid])[0, 1]
    else:
        quiet_r = np.nan

    results.append({
        'feature': feat_name,
        'pearson': pearson_r,
        'spearman': spearman_r,
        'storm_r': storm_r,
        'quiet_nz_r': quiet_r,
    })

corr_df = pd.DataFrame(results)
corr_df['abs_pearson'] = corr_df['pearson'].abs()
corr_df['abs_spearman'] = corr_df['spearman'].abs()
corr_df['abs_storm'] = corr_df['storm_r'].abs()
corr_df = corr_df.sort_values('abs_spearman', ascending=False).reset_index(drop=True)

# 保留 corr_series 兼容后续 cell
corr_series = corr_df.set_index('feature')['pearson'].sort_values(key=abs, ascending=False)

# ====== 可视化 ======
fig, axes = plt.subplots(1, 3, figsize=(20, 10))

# 图1: Pearson vs Spearman Top 30
top30 = corr_df.head(30)
y_pos = np.arange(len(top30))
ax = axes[0]
ax.barh(y_pos - 0.2, top30['pearson'], 0.35, label='Pearson', color='steelblue', alpha=0.8)
ax.barh(y_pos + 0.2, top30['spearman'], 0.35, label='Spearman', color='coral', alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top30['feature'], fontsize=8)
ax.set_xlabel('相关系数')
ax.set_title('Top 30 特征: Pearson vs Spearman')
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.5)
ax.legend()

# 图2: 风暴时段相关性 Top 20
storm_top = corr_df.dropna(subset=['storm_r']).nlargest(20, 'abs_storm')
ax = axes[1]
colors_bar = ['#F44336' if v > 0 else '#2196F3' for v in storm_top['storm_r']]
ax.barh(range(len(storm_top)), storm_top['storm_r'], color=colors_bar, alpha=0.8)
ax.set_yticks(range(len(storm_top)))
ax.set_yticklabels(storm_top['feature'], fontsize=8)
ax.set_xlabel('Pearson 相关系数 (仅风暴时段)')
ax.set_title('风暴时段 Top 20 特征')
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.5)

# 图3: Pearson vs Spearman 散点 (找分歧大的特征)
ax = axes[2]
ax.scatter(corr_df['pearson'], corr_df['spearman'], alpha=0.5, s=30)
# 标注分歧大的
diff = (corr_df['pearson'] - corr_df['spearman']).abs()
divergent = corr_df[diff > 0.03]
for _, row in divergent.iterrows():
    ax.annotate(row['feature'], (row['pearson'], row['spearman']), fontsize=7, alpha=0.7)
ax.plot([-0.3, 0.3], [-0.3, 0.3], 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('Pearson')
ax.set_ylabel('Spearman')
ax.set_title('Pearson vs Spearman 对比\n(偏离对角线 = 非线性/极端值影响)')
ax.legend()

plt.tight_layout()
plt.show()

# 打印 Top 20
print("=== Top 20 特征 (按 Spearman 排序) ===")
print(f"{'特征':>18s} | {'Pearson':>8s} | {'Spearman':>8s} | {'风暴时段':>8s} | {'日常(去零)':>10s}")
print("-" * 65)
for _, row in corr_df.head(20).iterrows():
    print(f"{row['feature']:>18s} | {row['pearson']:>+8.4f} | {row['spearman']:>+8.4f} | "
          f"{row['storm_r']:>+8.4f} | {row['quiet_nz_r']:>+10.4f}" 
          if not np.isnan(row['storm_r']) else
          f"{row['feature']:>18s} | {row['pearson']:>+8.4f} | {row['spearman']:>+8.4f} | {'N/A':>8s} | {'N/A':>10s}")

print(f"\n观察:")
print(f"  - Spearman 更能反映单调(非线性)关系，排名可能和 Pearson 不同")
print(f"  - 风暴时段的相关性结构可能与全局不同 — 对预测极端事件更有参考价值")

## B2: 共线性分析 + 自动去冗余

高度相关的特征（|r| > 0.9）携带几乎相同信息，留一个即可。
用贪心策略自动去冗余：从最相关特征开始，逐个加入，如果和已选特征共线性过高就跳过。

In [ ]:
# B2: 共线性分析 + 贪心去冗余

# 取 Spearman Top 40 做共线性分析
top40 = corr_df.head(40)
top40_names = list(top40['feature'])
top40_indices = [weather_features.index(n) for n in top40_names]

weather_flat = weather.reshape(-1, F)
top40_data = weather_flat[:, top40_indices]

# 处理 NaN: 用列均值填充
col_means = np.nanmean(top40_data, axis=0)
for j in range(top40_data.shape[1]):
    mask = np.isnan(top40_data[:, j])
    top40_data[mask, j] = col_means[j]

feat_corr_matrix = np.corrcoef(top40_data.T)

# ====== 热力图 ======
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

ax = axes[0]
sns.heatmap(feat_corr_matrix, xticklabels=top40_names, yticklabels=top40_names,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=False, ax=ax)
ax.set_title("Top 40 天气特征间的相关性矩阵")
ax.tick_params(axis='both', labelsize=7)

# ====== 贪心去冗余 ======
collinearity_threshold = 0.85
selected_features = []
rejected = {}

# 按 Spearman 相关性从高到低遍历
for _, row in corr_df.iterrows():
    feat = row['feature']
    fi = weather_features.index(feat)

    if not selected_features:
        selected_features.append(feat)
        continue

    # 算和已选特征的最大相关性
    selected_indices = [weather_features.index(f) for f in selected_features]
    feat_data = weather_flat[:, fi]
    max_corr = 0
    most_similar = None
    for sf in selected_features:
        sfi = weather_features.index(sf)
        sf_data = weather_flat[:, sfi]
        valid = ~(np.isnan(feat_data) | np.isnan(sf_data))
        if valid.sum() > 100:
            r = abs(np.corrcoef(feat_data[valid], sf_data[valid])[0, 1])
            if r > max_corr:
                max_corr = r
                most_similar = sf

    if max_corr < collinearity_threshold:
        selected_features.append(feat)
    else:
        rejected[feat] = (most_similar, max_corr)

print(f"=== 贪心去冗余结果 (阈值 |r| > {collinearity_threshold}) ===")
print(f"  原始特征: {len(corr_df)}")
print(f"  保留特征: {len(selected_features)}")
print(f"  去除冗余: {len(rejected)}")

# 可视化: 保留特征的相关性矩阵
sel_indices = [weather_features.index(f) for f in selected_features[:25]]
sel_data = weather_flat[:, sel_indices]
col_means_sel = np.nanmean(sel_data, axis=0)
for j in range(sel_data.shape[1]):
    mask = np.isnan(sel_data[:, j])
    sel_data[mask, j] = col_means_sel[j]
sel_corr = np.corrcoef(sel_data.T)

ax = axes[1]
sel_names_25 = selected_features[:25]
sns.heatmap(sel_corr, xticklabels=sel_names_25, yticklabels=sel_names_25,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=False, ax=ax)
ax.set_title(f"去冗余后 Top 25 特征的相关性矩阵\n(最大共线性 < {collinearity_threshold})")
ax.tick_params(axis='both', labelsize=7)

plt.tight_layout()
plt.show()

# 打印详情
print(f"\n=== 保留的特征 (前 25) ===")
for i, feat in enumerate(selected_features[:25]):
    row = corr_df[corr_df['feature'] == feat].iloc[0]
    print(f"  {i+1:2d}. {feat:18s} | Spearman={row['spearman']:+.4f} | Pearson={row['pearson']:+.4f}")

print(f"\n=== 被去除的冗余特征 (部分) ===")
for feat, (similar, r) in list(rejected.items())[:15]:
    print(f"  {feat:18s} → 与 {similar:18s} 共线性 r={r:.3f}")

## B3: 风暴事件"解剖" — 关键天气变量在风暴前后的变化

不再只看全时段时序对比，而是放大到每次风暴事件，观察：
- 天气变量在风暴前多少小时开始异常？（→ 预测窗口）
- 哪些变量是"先行指标"，在停电前就已经上升？

In [ ]:
# B3: 风暴事件解剖

# 选 top 6 特征 (去冗余后的)
top6_feats = selected_features[:6]
print(f"分析特征: {top6_feats}")

# 找最大的几次风暴事件 (按峰值排序)
storm_events = []
for s, e in zip(storm_starts, storm_ends):
    peak = total_outage_by_time[s:e].max()
    peak_idx = s + np.argmax(total_outage_by_time[s:e])
    storm_events.append({'start': s, 'end': e, 'peak': peak, 'peak_idx': peak_idx})
storm_events = sorted(storm_events, key=lambda x: x['peak'], reverse=True)

# 画 top 4 风暴
n_storms = min(4, len(storm_events))
fig, axes = plt.subplots(n_storms, 1, figsize=(18, 5 * n_storms))
if n_storms == 1:
    axes = [axes]

for si in range(n_storms):
    ev = storm_events[si]
    # 取风暴前后 24 小时的窗口
    window_start = max(0, ev['start'] - 24)
    window_end = min(T, ev['end'] + 24)
    t_slice = slice(window_start, window_end)
    t_range = timestamps[t_slice]
    n_pts = window_end - window_start

    ax = axes[si]
    # 停电曲线 (左 y 轴)
    ax.fill_between(t_range, total_outage_by_time[t_slice], alpha=0.3, color='black', label='停电总数')
    ax.plot(t_range, total_outage_by_time[t_slice], color='black', linewidth=1.5)
    ax.set_ylabel('停电户数', color='black')
    ax.axvspan(timestamps[ev['start']], timestamps[min(ev['end'], T-1)], alpha=0.1, color='red')
    ax.set_title(f"风暴 #{si+1}: {timestamps[ev['start']].strftime('%Y-%m-%d')} "
                 f"(峰值={ev['peak']:,.0f}, 持续={ev['end']-ev['start']}h)")

    # 天气变量 (右 y 轴, 标准化后叠加)
    ax2 = ax.twinx()
    colors_line = ['#E91E63', '#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#00BCD4']
    for fi_idx, feat_name in enumerate(top6_feats):
        fi = weather_features.index(feat_name)
        feat_vals = weather[t_slice, :, fi].mean(axis=1)  # 全县平均
        # z-score 标准化（用全局均值/标准差）
        global_mean = np.nanmean(weather[:, :, fi])
        global_std = np.nanstd(weather[:, :, fi])
        if global_std > 0:
            feat_z = (feat_vals - global_mean) / global_std
        else:
            feat_z = feat_vals * 0
        ax2.plot(t_range, feat_z, color=colors_line[fi_idx], linewidth=1.2,
                 alpha=0.8, label=feat_name)

    ax2.set_ylabel('天气变量 (z-score)')
    ax2.axhline(0, color='gray', linestyle=':', alpha=0.3)

    # 标注风暴开始线
    ax.axvline(timestamps[ev['start']], color='red', linestyle='--', alpha=0.5, label='风暴起止')
    if ev['end'] < T:
        ax.axvline(timestamps[ev['end']], color='red', linestyle='--', alpha=0.5)

    # 合并图例
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8, ncol=3)

axes[-1].set_xlabel('时间')
plt.tight_layout()
plt.show()

print("观察要点:")
print("  1. 哪些天气变量在风暴期间明显偏离零线 (z-score > 2)?")
print("  2. 这些变量是和停电同步上升，还是提前几小时就开始异常? (先行指标)")
print("  3. 不同风暴的天气 signature 是否一致?")

# ====== 量化: 各特征在风暴前的提前量 (交叉相关) ======
print(f"\n=== 交叉相关分析: 天气变量领先停电多少小时? ===")
print(f"{'特征':>18s} | {'最佳 lag(h)':>10s} | {'相关系数':>8s} | {'含义':>20s}")
print("-" * 65)
max_lag = 24
for feat_name in top6_feats:
    fi = weather_features.index(feat_name)
    feat_ts = weather[:, :, fi].mean(axis=1)  # 全县平均时序
    best_lag, best_r = 0, 0
    for lag in range(-max_lag, max_lag + 1):
        if lag >= 0:
            out_slice = total_outage_by_time[lag:]
            feat_slice = feat_ts[:T - lag]
        else:
            out_slice = total_outage_by_time[:T + lag]
            feat_slice = feat_ts[-lag:]
        valid = ~(np.isnan(out_slice) | np.isnan(feat_slice))
        if valid.sum() > 100:
            r = np.corrcoef(out_slice[valid], feat_slice[valid])[0, 1]
            if abs(r) > abs(best_r):
                best_r = r
                best_lag = lag
    meaning = f"天气领先停电 {best_lag}h" if best_lag > 0 else f"停电领先天气 {-best_lag}h" if best_lag < 0 else "同步"
    print(f"{feat_name:>18s} | {best_lag:>10d} | {best_r:>+8.4f} | {meaning:>20s}")

## B4: 缺失值检查

In [ ]:
# B4: 缺失值检查
print("=== 缺失值统计 ===")
print(f"  out 缺失: {np.isnan(outage).sum()} / {outage.size}")
print(f"  tracked 缺失: {np.isnan(tracked).sum()} / {tracked.size}")

weather_nan_per_feat = np.isnan(weather.reshape(-1, F)).sum(axis=0)
nan_df = pd.DataFrame({'feature': weather_features, 'nan_count': weather_nan_per_feat})
nan_df['nan_pct'] = nan_df['nan_count'] / (T * L) * 100
nan_with_missing = nan_df[nan_df['nan_count'] > 0].sort_values('nan_count', ascending=False)

if len(nan_with_missing) > 0:
    print(f"\n  有缺失值的天气特征 ({len(nan_with_missing)} 个):")
    print(nan_with_missing.to_string(index=False))
else:
    print("\n  所有天气特征均无缺失值")

## B5: 多方法共识投票 + 符号一致性检验

之前用 `(rf_rank + corr_rank)/2` 的问题：
1. 等权平均没有依据——为什么 RF 和 Spearman 该 1:1？
2. 忽略了符号一致性——如果 Pearson 说正相关但风暴时段说负相关，说明信号不稳定
3. 排名差距被抹平——排名差 2 和差 50 等同对待

**改进方案：共识投票 (Consensus Voting)**
- 每种方法独立投票：该方法 Top-K 的特征得 1 票，否则 0 票
- 最终得分 = 得票数。不拍权重，只看"有几种方法都认为它重要"
- 额外加**符号一致性**检查：多种方法方向一致 → 加分；方向矛盾 → 扣分/标记

In [ ]:
# B5: 多方法共识投票 + 符号一致性
from sklearn.ensemble import RandomForestRegressor

# ====== 1. 训练随机森林 ======
np.random.seed(42)
storm_time_idx = np.where(is_storm)[0]
quiet_time_idx = np.where(is_quiet)[0]
n_quiet_sample = min(len(quiet_time_idx), len(storm_time_idx) * 3)
quiet_sample_idx = np.random.choice(quiet_time_idx, n_quiet_sample, replace=False)
sample_time_idx = np.sort(np.concatenate([storm_time_idx, quiet_sample_idx]))
sample_locs = np.random.choice(L, min(10, L), replace=False)

X_rows, y_rows = [], []
for t in sample_time_idx:
    for li in sample_locs:
        w_vec = weather[t, li, :]
        if not np.any(np.isnan(w_vec)):
            X_rows.append(w_vec)
            y_rows.append(outage[t, li])

X_rf = np.array(X_rows)
y_rf = np.array(y_rows)
print(f"RF 训练样本: {X_rf.shape[0]:,} (特征数: {X_rf.shape[1]})")

rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=20,
                           n_jobs=-1, random_state=42)
rf.fit(X_rf, y_rf)

rf_importance = pd.Series(rf.feature_importances_, index=weather_features).sort_values(ascending=False)

# ====== 2. 构建多方法投票矩阵 ======
K = 25  # 每种方法取 Top-K 投票

# 四种方法各自的 Top-K 集合
method_rankings = {
    'Spearman':     corr_df.nlargest(K, 'abs_spearman')['feature'].tolist(),
    'Pearson':      corr_df.nlargest(K, 'abs_pearson')['feature'].tolist(),
    'Storm_Pearson':corr_df.dropna(subset=['storm_r']).nlargest(K, 'abs_storm')['feature'].tolist(),
    'RF':           rf_importance.head(K).index.tolist(),
}

# 投票矩阵
all_feats = weather_features
vote_df = pd.DataFrame({'feature': all_feats})
for method, top_list in method_rankings.items():
    vote_df[method] = vote_df['feature'].isin(top_list).astype(int)
vote_df['votes'] = vote_df[list(method_rankings.keys())].sum(axis=1)

# ====== 3. 符号一致性检查 ======
# 收集各方法的符号信息 (正/负/不确定)
sign_info = {}
for feat in all_feats:
    signs = []
    row = corr_df[corr_df['feature'] == feat]
    if len(row) > 0:
        row = row.iloc[0]
        if abs(row['pearson']) > 0.005:
            signs.append(np.sign(row['pearson']))
        if abs(row['spearman']) > 0.005:
            signs.append(np.sign(row['spearman']))
        if not np.isnan(row['storm_r']) and abs(row['storm_r']) > 0.005:
            signs.append(np.sign(row['storm_r']))
        if not np.isnan(row['quiet_nz_r']) and abs(row['quiet_nz_r']) > 0.005:
            signs.append(np.sign(row['quiet_nz_r']))
    # RF 没有符号，跳过
    if len(signs) >= 2:
        # 一致性 = 多数符号的占比
        pos = sum(1 for s in signs if s > 0)
        neg = sum(1 for s in signs if s < 0)
        majority = max(pos, neg)
        sign_info[feat] = {
            'sign_consistency': majority / len(signs),
            'dominant_sign': '+' if pos >= neg else '-',
            'n_methods': len(signs),
            'conflict': pos > 0 and neg > 0,
        }
    else:
        sign_info[feat] = {'sign_consistency': 1.0, 'dominant_sign': '?', 'n_methods': len(signs), 'conflict': False}

sign_df = pd.DataFrame(sign_info).T
sign_df.index.name = 'feature'
sign_df = sign_df.reset_index()

# 合并
vote_df = vote_df.merge(sign_df, on='feature', how='left')
vote_df['sign_consistency'] = vote_df['sign_consistency'].fillna(1.0)
vote_df['conflict'] = vote_df['conflict'].fillna(False)

# 排序: 先按票数，再按符号一致性
vote_df = vote_df.sort_values(['votes', 'sign_consistency'], ascending=[False, False]).reset_index(drop=True)

# ====== 4. 可视化 ======
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# 图1: 投票矩阵热力图 (Top 30)
top30_vote = vote_df.head(30)
ax = axes[0]
matrix_data = top30_vote[list(method_rankings.keys())].values
sns.heatmap(matrix_data, ax=ax, cmap='YlGn', cbar=False, linewidths=0.5,
            xticklabels=list(method_rankings.keys()),
            yticklabels=top30_vote['feature'], annot=True, fmt='d')
# 在右侧标注总票数
for i, (_, row) in enumerate(top30_vote.iterrows()):
    conflict_marker = ' ⚠' if row['conflict'] else ''
    ax.text(len(method_rankings) + 0.3, i + 0.5,
            f"{row['votes']:.0f}票 ({row['dominant_sign']}){conflict_marker}",
            va='center', fontsize=8,
            color='red' if row['conflict'] else 'black')
ax.set_title(f'多方法投票矩阵 (每方法 Top-{K})\n1=该方法认为重要, ⚠=符号冲突')

# 图2: RF 重要性 vs Spearman 排名
ax = axes[1]
rf_imp_df = pd.DataFrame({'feature': weather_features, 'rf_imp': rf.feature_importances_})
rf_imp_df = rf_imp_df.merge(corr_df[['feature', 'abs_spearman']], on='feature')
rf_imp_df = rf_imp_df.merge(vote_df[['feature', 'votes', 'conflict']], on='feature')
colors_scatter = ['red' if c else ('#4CAF50' if v >= 3 else '#2196F3' if v >= 2 else '#bdbdbd')
                  for v, c in zip(rf_imp_df['votes'], rf_imp_df['conflict'])]
ax.scatter(rf_imp_df['abs_spearman'], rf_imp_df['rf_imp'], c=colors_scatter, s=40, alpha=0.7)

# 标注高票特征
high_vote = rf_imp_df[rf_imp_df['votes'] >= 3]
for _, row in high_vote.iterrows():
    ax.annotate(row['feature'], (row['abs_spearman'], row['rf_imp']),
                fontsize=7, alpha=0.8)
ax.set_xlabel('|Spearman 相关系数|')
ax.set_ylabel('RF 特征重要性')
ax.set_title('线性 vs 非线性重要性\n(绿=≥3票共识, 蓝=2票, 灰=≤1票, 红=符号冲突)')

# 图3: 符号一致性分布
ax = axes[2]
# 只看有投票的特征
voted = vote_df[vote_df['votes'] >= 1].copy()
conflict_counts = voted.groupby('votes')['conflict'].agg(['sum', 'count'])
conflict_counts.columns = ['有冲突', '总数']
conflict_counts['无冲突'] = conflict_counts['总数'] - conflict_counts['有冲突']
conflict_counts[['无冲突', '有冲突']].plot(kind='bar', stacked=True, ax=ax,
                                        color=['#4CAF50', '#F44336'], alpha=0.8)
ax.set_xlabel('得票数')
ax.set_ylabel('特征数量')
ax.set_title('各票数档位的符号一致性')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# ====== 5. 最终输出 ======
print("=" * 85)
print(f"多方法共识投票结果 (每方法 Top-{K}, 共 {len(method_rankings)} 种方法)")
print("=" * 85)
print(f"{'排名':>4s} | {'特征':>18s} | {'Spear':>5s} {'Pears':>5s} {'Storm':>5s} {'RF':>5s} | "
      f"{'总票':>4s} | {'符号':>4s} | {'一致性':>5s} | {'冲突':>4s}")
print("-" * 85)
for i, (_, row) in enumerate(vote_df[vote_df['votes'] >= 2].iterrows()):
    print(f"{i+1:>4d} | {row['feature']:>18s} | "
          f"{'✓' if row['Spearman'] else '·':>5s} "
          f"{'✓' if row['Pearson'] else '·':>5s} "
          f"{'✓' if row['Storm_Pearson'] else '·':>5s} "
          f"{'✓' if row['RF'] else '·':>5s} | "
          f"{row['votes']:>4.0f} | {row['dominant_sign']:>4s} | "
          f"{row['sign_consistency']:>5.0%} | "
          f"{'⚠ 是' if row['conflict'] else '  否':>4s}")

# 最终推荐: 去冗余后的 + ≥2票 + 无符号冲突
consensus_feats = set(vote_df[(vote_df['votes'] >= 2) & (~vote_df['conflict'])]['feature'])
final_selected = [f for f in selected_features if f in consensus_feats]
# 补充共识高但被去冗余删掉的 (≥3票)
strong_consensus = set(vote_df[(vote_df['votes'] >= 3) & (~vote_df['conflict'])]['feature'])
bonus = [f for f in strong_consensus if f not in final_selected]
final_selected.extend(bonus)

print(f"\n=== 最终推荐特征 ({len(final_selected)} 个) ===")
print(f"  筛选条件: ≥2 方法共识 + 无符号冲突 + 共线性去冗余")
for i, f in enumerate(final_selected):
    v = vote_df[vote_df['feature'] == f]['votes'].values[0]
    s = vote_df[vote_df['feature'] == f]['dominant_sign'].values[0]
    print(f"  {i+1:2d}. {f:18s} ({v:.0f}票, 方向{s})")

# 标记符号冲突的特征供讨论
conflicts = vote_df[(vote_df['votes'] >= 2) & (vote_df['conflict'])]
if len(conflicts) > 0:
    print(f"\n=== ⚠ 符号冲突特征 (需人工判断) ===")
    print(f"  这些特征多方法都认为重要，但方向不一致:")
    for _, row in conflicts.iterrows():
        r = corr_df[corr_df['feature'] == row['feature']].iloc[0]
        print(f"  {row['feature']:18s} | Pearson={r['pearson']:+.4f} Storm={r['storm_r']:+.4f} "
              f"Quiet={r['quiet_nz_r']:+.4f}")
    print(f"  → 方向矛盾可能是因为: 风暴vs日常行为不同、非线性关系、或特征本身含义复杂")
    print(f"  → 建议: 仍然保留这些特征让模型自己学，但不要用符号做硬规则")
else:
    print(f"\n  所有共识特征符号一致，无冲突。")

print(f"\n方法论总结:")
print(f"  1. 不人为定权重 — 用投票制，每种方法有平等的 1 票")
print(f"  2. 符号一致性作为额外信号 — 冲突不代表不重要，只是需要注意")
print(f"  3. 最终特征列表同时通过: 多方法共识 + 共线性去冗余 + 符号检查")

---
# Task C: 通用特征构造

**目标**：整合 Part A / B 的全部发现，构造一套通用特征管线，后续所有模型共享。

**基于前两部分的关键设计决策**：

| Part A/B 发现 | → 特征设计 |
|---|---|
| 数据 95%+ 为零，极端事件主导 RMSE | → 停电滞后/滚动统计 + 二值化"是否在停电" |
| GMM 分出 5 个停电层级 | → 用 GMM 阈值构造分层指示特征 |
| 风暴 vs 日常是两种不同过程 | → "近期是否有风暴迹象"指示特征 (基于天气) |
| B3 交叉相关: 天气变量有数小时领先量 | → 天气 lag 特征 (1h, 3h, 6h) |
| B5 共识投票: 筛出最终特征列表 | → 只对共识特征构造高阶衍生 |
| B2 去冗余: 原始 109 → 精简集 | → 天气原始值只用精简集 |
| 日常时段去零后有弱日内周期 | → sin/cos 时间编码保留 |
| 停电率 = out/tracked 更有意义 | → 加 outage_rate 特征 |

In [ ]:
# Task C: 通用特征构造函数
# 所有特征只用 t 及之前的信息 (shift ≥ 1)，严格防止数据泄漏

def build_features(ds, 
                   selected_weather_features=None,
                   top_weather_features=None,
                   gmm_thresholds=None,
                   weather_lags=(1, 3, 6),
                   outage_lags=(1, 3, 6, 12, 24),
                   rolling_windows=(6, 12, 24)):
    """
    从 xarray dataset 构造特征 DataFrame。
    
    参数:
        ds: xarray Dataset
        selected_weather_features: B2 去冗余后的天气特征 (原始值)
        top_weather_features: B5 共识投票 top 特征 (额外构造 lag/rolling/交互)
        gmm_thresholds: A2 的 GMM 阈值列表 (用于分层指示特征)
        weather_lags: 天气 lag 步长 (仅对 top 特征)
        outage_lags: 停电 lag 步长
        rolling_windows: 滚动窗口大小
    
    返回:
        df: DataFrame，每行 = (timestamp, location)
        feature_groups: dict, 各类特征名列表 (方便后续分析)
    """
    timestamps = pd.to_datetime(ds.timestamp.values)
    locations = list(ds.location.values)
    all_features = list(ds.feature.values)
    
    out = ds.out.transpose("timestamp", "location").values.astype(float)
    trk = ds.tracked.transpose("timestamp", "location").values.astype(float)
    w = ds.weather.transpose("timestamp", "location", "feature").values.astype(float)
    
    T, L = out.shape
    
    # 确定天气特征子集
    if selected_weather_features is None:
        selected_weather_features = all_features
    if top_weather_features is None:
        top_weather_features = selected_weather_features[:6]
    
    sel_indices = {f: all_features.index(f) for f in selected_weather_features}
    top_indices = {f: all_features.index(f) for f in top_weather_features if f in all_features}
    
    # 特征组追踪
    feat_groups = {
        'target': ['out', 'tracked'],
        'time': [], 'outage_lag': [], 'outage_rolling': [],
        'outage_regime': [], 'weather_raw': [], 'weather_lag': [],
        'weather_rolling': [], 'weather_interaction': [], 'storm_indicator': []
    }
    
    records = []
    
    for li in range(L):
        loc = locations[li]
        df_loc = pd.DataFrame({
            'timestamp': timestamps,
            'location': loc,
            'out': out[:, li],
            'tracked': trk[:, li],
        })
        
        # ================================================================
        # 1. 时间特征
        # ================================================================
        df_loc['hour'] = timestamps.hour
        df_loc['day_of_week'] = timestamps.dayofweek
        df_loc['month'] = timestamps.month
        df_loc['hour_sin'] = np.sin(2 * np.pi * timestamps.hour / 24)
        df_loc['hour_cos'] = np.cos(2 * np.pi * timestamps.hour / 24)
        df_loc['dow_sin'] = np.sin(2 * np.pi * timestamps.dayofweek / 7)
        df_loc['dow_cos'] = np.cos(2 * np.pi * timestamps.dayofweek / 7)
        df_loc['month_sin'] = np.sin(2 * np.pi * timestamps.month / 12)
        df_loc['month_cos'] = np.cos(2 * np.pi * timestamps.month / 12)
        
        if li == 0:
            feat_groups['time'] = ['hour', 'day_of_week', 'month',
                                   'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
                                   'month_sin', 'month_cos']
        
        # ================================================================
        # 2. 停电历史特征 (全部 shift ≥ 1)
        # ================================================================
        out_series = df_loc['out']
        
        # 2a. 停电率
        df_loc['outage_rate'] = df_loc['out'] / df_loc['tracked'].replace(0, np.nan)
        
        # 2b. lag 特征
        for lag in outage_lags:
            df_loc[f'out_lag_{lag}h'] = out_series.shift(lag)
            df_loc[f'rate_lag_{lag}h'] = df_loc['outage_rate'].shift(lag)
        
        # 2c. 滚动窗口统计 (shift(1) 防止泄漏)
        shifted_out = out_series.shift(1)
        for win in rolling_windows:
            roll = shifted_out.rolling(win, min_periods=1)
            df_loc[f'out_rmean_{win}h'] = roll.mean()
            df_loc[f'out_rmax_{win}h'] = roll.max()
            df_loc[f'out_rstd_{win}h'] = roll.std().fillna(0)
            df_loc[f'out_rsum_{win}h'] = roll.sum()
        
        # 2d. 二值化: 过去 N 小时是否有停电
        for win in [6, 12, 24]:
            df_loc[f'had_outage_{win}h'] = (shifted_out.rolling(win, min_periods=1).max() > 0).astype(int)
        
        # 2e. GMM 分层指示 (A2 的阈值)
        if gmm_thresholds is not None and len(gmm_thresholds) > 0:
            last_out = out_series.shift(1)
            # 上一步的停电所属层级 (ordinal编码: 0=零, 1=最低层, ..., k=最高层)
            regime = pd.Series(np.zeros(T), index=df_loc.index)
            for ti, thresh in enumerate(gmm_thresholds):
                regime = regime + (last_out > thresh).astype(int)
            df_loc['out_regime'] = regime
            # 过去24h最高层级
            df_loc['out_regime_max_24h'] = df_loc['out_regime'].rolling(24, min_periods=1).max()
            
            if li == 0:
                feat_groups['outage_regime'] = ['out_regime', 'out_regime_max_24h']
        
        if li == 0:
            feat_groups['outage_lag'] = [f'out_lag_{l}h' for l in outage_lags] + \
                                        [f'rate_lag_{l}h' for l in outage_lags]
            feat_groups['outage_rolling'] = []
            for win in rolling_windows:
                feat_groups['outage_rolling'] += [f'out_rmean_{win}h', f'out_rmax_{win}h', 
                                                   f'out_rstd_{win}h', f'out_rsum_{win}h']
            feat_groups['outage_rolling'] += [f'had_outage_{w}h' for w in [6, 12, 24]]
        
        # ================================================================
        # 3. 天气特征 — 原始值 (去冗余后的精简集)
        # ================================================================
        for fname, fi in sel_indices.items():
            df_loc[f'w_{fname}'] = w[:, li, fi]
        
        if li == 0:
            feat_groups['weather_raw'] = [f'w_{f}' for f in sel_indices]
        
        # ================================================================
        # 4. 天气 lag 特征 (仅 top 共识特征, 基于 B3 交叉相关)
        # ================================================================
        for fname, fi in top_indices.items():
            w_series = pd.Series(w[:, li, fi])
            for lag in weather_lags:
                df_loc[f'w_{fname}_lag{lag}h'] = w_series.shift(lag)
        
        if li == 0:
            feat_groups['weather_lag'] = [f'w_{f}_lag{l}h' for f in top_indices for l in weather_lags]
        
        # ================================================================
        # 5. 天气滚动统计 (仅 top 特征)
        # ================================================================
        for fname, fi in top_indices.items():
            w_series = pd.Series(w[:, li, fi])
            for win in rolling_windows:
                roll = w_series.rolling(win, min_periods=1)
                df_loc[f'w_{fname}_rmean_{win}h'] = roll.mean()
                df_loc[f'w_{fname}_rmax_{win}h'] = roll.max()
            # 变化率: 当前值 vs 6h前
            df_loc[f'w_{fname}_delta6h'] = w_series - w_series.shift(6)
        
        if li == 0:
            feat_groups['weather_rolling'] = []
            for f in top_indices:
                for win in rolling_windows:
                    feat_groups['weather_rolling'] += [f'w_{f}_rmean_{win}h', f'w_{f}_rmax_{win}h']
                feat_groups['weather_rolling'].append(f'w_{f}_delta6h')
        
        # ================================================================
        # 6. 天气交互特征 (top 特征两两乘积, 只取前 3 个)
        # ================================================================
        top_list = list(top_indices.keys())[:4]
        interaction_names = []
        for i in range(len(top_list)):
            for j in range(i + 1, len(top_list)):
                f1, f2 = top_list[i], top_list[j]
                col_name = f'w_{f1}_x_{f2}'
                df_loc[col_name] = df_loc[f'w_{f1}'] * df_loc[f'w_{f2}']
                interaction_names.append(col_name)
        
        if li == 0:
            feat_groups['weather_interaction'] = interaction_names
        
        # ================================================================
        # 7. 风暴迹象指示特征 (A4: 基于天气判断是否可能进入风暴)
        # ================================================================
        # 用 top 特征的 z-score 综合打分: 多个天气变量同时异常 → 风暴可能性高
        if len(top_indices) >= 3:
            z_scores = []
            for fname, fi in list(top_indices.items())[:5]:
                w_col = w[:, li, fi]
                w_mean = np.nanmean(w_col)
                w_std = np.nanstd(w_col)
                if w_std > 0:
                    z = (w_col - w_mean) / w_std
                else:
                    z = np.zeros(T)
                z_scores.append(z)
            
            z_matrix = np.array(z_scores)  # (n_feats, T)
            # 同时异常指标: 有几个天气变量 |z| > 1.5
            df_loc['n_weather_anomaly'] = (np.abs(z_matrix) > 1.5).sum(axis=0)
            # 综合异常强度: z-score 绝对值的均值
            df_loc['weather_anomaly_score'] = np.abs(z_matrix).mean(axis=0)
            # 过去 6h 最大异常分数
            df_loc['anomaly_score_max_6h'] = df_loc['weather_anomaly_score'].rolling(6, min_periods=1).max()
            
            if li == 0:
                feat_groups['storm_indicator'] = ['n_weather_anomaly', 'weather_anomaly_score', 
                                                   'anomaly_score_max_6h']
        
        records.append(df_loc)
    
    df = pd.concat(records, ignore_index=True)
    return df, feat_groups

print("build_features() 函数定义完成")
print("返回: (df, feature_groups)")
print("  df: 每行 = (timestamp, location) 的完整特征矩阵")
print("  feature_groups: dict, 各类特征名列表，方便按组分析/消融")

In [ ]:
# Task C: 运行特征构造 + 质量检查

# 使用 Part A/B 的全部产出
print("=" * 60)
print("特征构造参数 (来自 Part A/B)")
print("=" * 60)
print(f"  B2 去冗余特征 (天气原始值): {len(selected_features)} 个")
print(f"  B5 共识投票 top 特征 (高阶衍生): {len(final_selected)} 个")
print(f"  A2 GMM 阈值: {[f'{t:.0f}' for t in gmm_thresholds]}")

df, feat_groups = build_features(
    ds,
    selected_weather_features=selected_features,
    top_weather_features=final_selected[:6],  # top 6 做高阶衍生
    gmm_thresholds=gmm_thresholds,
)

# ====== 1. 基本信息 ======
print(f"\n{'=' * 60}")
print(f"构造结果")
print(f"{'=' * 60}")
print(f"  DataFrame: {df.shape[0]:,} 行 × {df.shape[1]} 列")
print(f"  内存占用: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# ====== 2. 按特征组统计 ======
print(f"\n  特征分组明细:")
total_feats = 0
for group, names in feat_groups.items():
    if group == 'target':
        continue
    print(f"    {group:25s}: {len(names):3d} 个")
    total_feats += len(names)
print(f"    {'─' * 35}")
print(f"    {'总计 (不含 target)':25s}: {total_feats:3d} 个")

# ====== 3. NaN 检查 ======
nan_counts = df.isnull().sum()
nan_cols = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(f"\n  有 NaN 的列: {len(nan_cols)} 个 (lag/rolling 前几行有 NaN 是正常的)")
if len(nan_cols) > 0:
    print(f"    最大 NaN 数: {nan_cols.iloc[0]:,} (共 {df.shape[0]:,} 行, "
          f"占 {nan_cols.iloc[0]/df.shape[0]*100:.1f}%)")
    # 检查是否有非 lag 导致的意外 NaN
    unexpected_nan = nan_cols[nan_cols > L * 24]  # 超过 24h * L 个县的 NaN 不正常
    if len(unexpected_nan) > 0:
        print(f"    ⚠ 以下列 NaN 数量异常，需要排查:")
        for col, cnt in unexpected_nan.items():
            print(f"      {col}: {cnt:,}")
    else:
        print(f"    ✓ 所有 NaN 都在预期范围内 (lag/rolling 的初始行)")

# ====== 4. 数据泄漏检查 ======
print(f"\n  数据泄漏检查:")
# 验证: 每个 lag 特征在 t=0 时应该是 NaN (无法从未来获取信息)
sample_loc = df[df['location'] == df['location'].iloc[0]]
leak_check_passed = True
for lag in [1, 3, 6]:
    col = f'out_lag_{lag}h'
    if col in df.columns:
        first_vals = sample_loc[col].iloc[:lag]
        if first_vals.notna().any():
            print(f"    ⚠ {col} 的前 {lag} 行应为 NaN 但不是!")
            leak_check_passed = False
if leak_check_passed:
    print(f"    ✓ 所有 lag 特征的前 N 行均为 NaN，无数据泄漏")

# ====== 5. 特征分布概览 ======
print(f"\n  各组特征均值范围 (检查数量级是否合理):")
for group, names in feat_groups.items():
    if group == 'target' or len(names) == 0:
        continue
    cols_exist = [c for c in names if c in df.columns]
    if cols_exist:
        means = df[cols_exist].mean()
        print(f"    {group:25s}: min_mean={means.min():>10.3f}, max_mean={means.max():>10.3f}")

# ====== 6. 风暴指示特征验证 ======
if 'weather_anomaly_score' in df.columns:
    print(f"\n  风暴指示特征验证:")
    # 在已知风暴时段，异常分数应该明显更高
    time_to_idx = {t: i for i, t in enumerate(timestamps)}
    storm_mask_df = df['timestamp'].isin(timestamps[is_storm])
    quiet_mask_df = ~storm_mask_df
    
    storm_score = df.loc[storm_mask_df, 'weather_anomaly_score'].mean()
    quiet_score = df.loc[quiet_mask_df, 'weather_anomaly_score'].mean()
    print(f"    风暴时段异常分数均值: {storm_score:.3f}")
    print(f"    日常时段异常分数均值: {quiet_score:.3f}")
    print(f"    倍数: {storm_score/quiet_score:.1f}x → {'✓ 有区分度' if storm_score > quiet_score * 1.5 else '⚠ 区分度不够'}")

print(f"\n{'=' * 60}")
print(f"特征矩阵已就绪，可供后续 Task D 及所有模型使用")
print(f"{'=' * 60}")

In [ ]:
# C2: 特征质量可视化

# ====== 1. 各组特征与 out 的相关性分布 ======
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 左图: 每组特征和 out 的 Pearson 相关性箱线图
ax = axes[0]
group_corrs = {}
for group, names in feat_groups.items():
    if group == 'target' or len(names) == 0:
        continue
    cols = [c for c in names if c in df.columns]
    if cols:
        corrs = df[cols].corrwith(df['out']).dropna().values
        if len(corrs) > 0:
            group_corrs[group] = corrs

group_names = list(group_corrs.keys())
group_data = [group_corrs[g] for g in group_names]
bp = ax.boxplot(group_data, labels=group_names, patch_artist=True, showfliers=True)
colors_bp = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0', '#E91E63', '#00BCD4', '#F44336', '#795548', '#607D8B']
for patch, color in zip(bp['boxes'], colors_bp[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('与 out 的 Pearson 相关系数')
ax.set_title('各组特征与停电的相关性分布')
ax.tick_params(axis='x', rotation=30)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')

# 右图: 各组中相关性最强的特征
ax = axes[1]
top_per_group = []
for group, names in feat_groups.items():
    if group == 'target' or len(names) == 0:
        continue
    cols = [c for c in names if c in df.columns]
    if cols:
        corrs = df[cols].corrwith(df['out']).dropna()
        if len(corrs) > 0:
            best_col = corrs.abs().idxmax()
            top_per_group.append({
                'group': group, 'feature': best_col,
                'corr': corrs[best_col]
            })

top_df = pd.DataFrame(top_per_group).sort_values('corr', key=abs, ascending=True)
bar_colors = ['#F44336' if v > 0 else '#2196F3' for v in top_df['corr']]
ax.barh(range(len(top_df)), top_df['corr'], color=bar_colors, alpha=0.8)
ax.set_yticks(range(len(top_df)))
ax.set_yticklabels([f"{row['group']}\n({row['feature']})" for _, row in top_df.iterrows()], fontsize=8)
ax.set_xlabel('Pearson 相关系数')
ax.set_title('各组中与停电最相关的单个特征')
ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

# ====== 2. 风暴指示特征效果验证 ======
if 'weather_anomaly_score' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 用一个县做 case study
    sample_fips = county_stats.iloc[0]['fips']  # 停电最严重的县
    df_sample = df[df['location'] == sample_fips].copy()
    ts = df_sample['timestamp']
    
    # 图1: 停电 vs 异常分数时序
    ax = axes[0]
    ax.plot(ts, df_sample['out'], color='black', linewidth=0.8, alpha=0.7, label='停电数')
    ax2 = ax.twinx()
    ax2.plot(ts, df_sample['weather_anomaly_score'], color='red', linewidth=0.8, alpha=0.7, label='异常分数')
    ax.set_ylabel('停电户数')
    ax2.set_ylabel('天气异常分数', color='red')
    ax.set_title(f'县 {sample_fips}: 停电 vs 天气异常分数')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
    
    # 图2: GMM 层级 vs 停电
    if 'out_regime' in df_sample.columns:
        ax = axes[1]
        for regime_val in sorted(df_sample['out_regime'].dropna().unique()):
            mask = df_sample['out_regime'] == regime_val
            ax.scatter(df_sample.loc[mask, 'weather_anomaly_score'],
                      df_sample.loc[mask, 'out'],
                      alpha=0.3, s=10, label=f'层级 {int(regime_val)}')
        ax.set_xlabel('天气异常分数')
        ax.set_ylabel('停电户数')
        ax.set_title('天气异常分数 vs 停电 (按 GMM 层级着色)')
        ax.set_yscale('symlog')
        ax.legend(fontsize=8)
    
    # 图3: 特征重要性排行 (快速 RF)
    ax = axes[2]
    all_feat_cols = []
    for group, names in feat_groups.items():
        if group != 'target':
            all_feat_cols.extend([c for c in names if c in df.columns])
    
    # 采样 + 快速 RF
    sample_df = df.dropna(subset=all_feat_cols[:20] + ['out']).sample(
        min(50000, len(df)), random_state=42)
    X_quick = sample_df[all_feat_cols[:20]].values  # 前 20 个特征
    y_quick = sample_df['out'].values
    
    from sklearn.ensemble import RandomForestRegressor
    rf_quick = RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
    rf_quick.fit(X_quick, y_quick)
    
    imp = pd.Series(rf_quick.feature_importances_, index=all_feat_cols[:20]).sort_values(ascending=True)
    imp.tail(15).plot(kind='barh', ax=ax, color='#4CAF50', alpha=0.8)
    ax.set_xlabel('RF 重要性')
    ax.set_title('构造特征重要性 Top 15 (快速验证)')
    
    plt.tight_layout()
    plt.show()

# ====== 3. 特征组总结 ======
print("=" * 60)
print("特征组速查表")
print("=" * 60)
for group, names in feat_groups.items():
    if group == 'target' or len(names) == 0:
        continue
    cols = [c for c in names if c in df.columns]
    if cols:
        corrs = df[cols].corrwith(df['out']).dropna()
        best = corrs.abs().idxmax() if len(corrs) > 0 else 'N/A'
        best_r = corrs[best] if best != 'N/A' else 0
        print(f"\n  {group} ({len(names)} 个):")
        print(f"    最强特征: {best} (r={best_r:+.4f})")
        print(f"    来源: ", end="")
        if 'time' in group:
            print("日历时间 sin/cos 编码")
        elif 'outage_lag' in group:
            print(f"停电历史 lag {outage_lags}")
        elif 'outage_rolling' in group:
            print(f"停电滚动窗口 {rolling_windows} + 二值化")
        elif 'regime' in group:
            print(f"A2 GMM 阈值 {[f'{t:.0f}' for t in gmm_thresholds]}")
        elif 'weather_raw' in group:
            print(f"B2 去冗余后的 {len(selected_features)} 个天气变量")
        elif 'weather_lag' in group:
            print(f"B3 交叉相关 → top 特征 lag {weather_lags}")
        elif 'weather_rolling' in group:
            print(f"top 特征滚动均值/最大值 + 6h变化率")
        elif 'interaction' in group:
            print(f"top 4 特征两两乘积")
        elif 'storm' in group:
            print(f"A4 z-score 综合异常指标")

## C2: 特征质量可视化 — 各组特征和停电的关系

---
# Task D: Baseline 复现 & 评估框架

**目标**：跑通 baseline，记录 RMSE，搭建统一评估函数。

## D1: 评估函数

统一的评估函数，所有模型都用这个来算分，方便对比。

In [ ]:
# D1: 统一评估函数
def evaluate_model(y_true, y_pred, locations, model_name="Model"):
    """
    评估模型预测结果。
    
    参数:
        y_true: (T, L) 真实停电数，T=时间步数, L=县数量
        y_pred: (T, L) 预测停电数
        locations: 县名列表
        model_name: 模型名称（用于打印）
    
    返回:
        overall_rmse: 全局平均 RMSE
        county_rmses: dict, {county: rmse}
    """
    county_rmses = {}
    for li, loc in enumerate(locations):
        diff = y_true[:, li] - y_pred[:, li]
        county_rmses[loc] = np.sqrt(np.mean(diff**2))
    
    overall_rmse = np.mean(list(county_rmses.values()))
    
    print(f"=== {model_name} ===")
    print(f"  Overall RMSE (县均): {overall_rmse:.4f}")
    
    # Top 5 最差的县
    sorted_counties = sorted(county_rmses.items(), key=lambda x: x[1], reverse=True)
    print(f"  RMSE 最高的 5 个县:")
    for loc, rmse in sorted_counties[:5]:
        print(f"    {loc}: {rmse:.4f}")
    
    return overall_rmse, county_rmses


def compare_models(results_dict):
    """
    对比多个模型的结果。
    
    参数:
        results_dict: {model_name: overall_rmse}
    """
    print("\n" + "="*50)
    print("模型对比")
    print("="*50)
    df = pd.DataFrame([
        {'Model': name, 'RMSE': rmse} 
        for name, rmse in sorted(results_dict.items(), key=lambda x: x[1])
    ])
    print(df.to_string(index=False))
    print(f"\n当前最优: {df.iloc[0]['Model']} (RMSE={df.iloc[0]['RMSE']:.4f})")

print("evaluate_model() 和 compare_models() 函数定义完成")

## D2: Persistence Baseline

最简单的 baseline：用每个县最后一个观测值作为未来所有时刻的预测。这是"什么都不做"的下限。

In [ ]:
# D2: Persistence baseline + 验证集评估
# 数据划分：前 80% 训练，后 20% 验证（和 demo 一致）
VALIDATION_SPLIT = 0.2
split_idx = int(T * (1 - VALIDATION_SPLIT))

train_outage = outage[:split_idx, :]   # (T_train, L)
val_outage = outage[split_idx:, :]     # (T_val, L)
val_timestamps = timestamps[split_idx:]

print(f"训练集: {split_idx} 小时 ({timestamps[0].date()} ~ {timestamps[split_idx-1].date()})")
print(f"验证集: {T - split_idx} 小时 ({timestamps[split_idx].date()} ~ {timestamps[-1].date()})")

# --- Persistence baseline ---
# 24h: 用训练集最后一个值重复 24 次
# 48h: 用训练集最后一个值重复 48 次

val_24h = val_outage[:24, :]  # 验证集前 24 小时
val_48h = val_outage[:48, :]  # 验证集前 48 小时

last_val = train_outage[-1, :]  # 训练集最后一个时刻的停电数 (L,)

persistence_24h = np.tile(last_val, (24, 1))  # (24, L)
persistence_48h = np.tile(last_val, (48, 1))  # (48, L)

# 评估
results = {}
rmse_24, _ = evaluate_model(val_24h, persistence_24h, locations, "Persistence 24h")
rmse_48, _ = evaluate_model(val_48h, persistence_48h, locations, "Persistence 48h")
results['Persistence_24h'] = rmse_24
results['Persistence_48h'] = rmse_48

# --- Historical Average baseline ---
# 用训练集每个县的全局平均值作为预测
hist_avg = train_outage.mean(axis=0)  # (L,)
hist_avg_24h = np.tile(hist_avg, (24, 1))
hist_avg_48h = np.tile(hist_avg, (48, 1))

print()
rmse_24_ha, _ = evaluate_model(val_24h, hist_avg_24h, locations, "Historical Average 24h")
rmse_48_ha, _ = evaluate_model(val_48h, hist_avg_48h, locations, "Historical Average 48h")
results['HistAvg_24h'] = rmse_24_ha
results['HistAvg_48h'] = rmse_48_ha

print("\n--- 后续: 跑完 demo.ipynb 的 SARIMAX 和 Seq2Seq 后把它们的 RMSE 也填进来 ---")
# results['SARIMAX_24h'] = xxx
# results['SARIMAX_48h'] = xxx
# results['Seq2Seq_24h'] = xxx
# results['Seq2Seq_48h'] = xxx
# compare_models(results)

---
# Phase 1 总结 & 下一步

跑完这个 notebook 后，你应该知道：

1. **数据特点**：停电数据极度稀疏，极端值由天气事件驱动
2. **重要特征**：从 109 个天气变量筛选到了 ~15 个最相关的
3. **特征工程**：`build_features()` 函数可以直接给后续模型用
4. **Baseline 水平**：Persistence / Historical Average / SARIMAX / Seq2Seq 的 RMSE 已知

**Phase 2 的方向**：
- 尝试更强的模型（XGBoost, LightGBM, Transformer 等）
- 用 `build_features()` 的输出作为起点，根据模型特点进一步调整
- 模型集成（Ensemble）
- Part II: 基于预测结果做发电机部署决策